# 8단계 · 점으로 보는 비지도 결과 — 경계선은 무엇을 나누나  [외부 데이터 사용 — B층]

표의 숫자(AP, TP@5%)만으로는 "비지도 모델이 정말 제대로 한 건가?"가 잘 안 보입니다.
쇼트 하나를 **점 하나**로 찍어서 직접 확인합니다.

- 점수: 5단계 **과거기준선**(원래 단위, 그 쇼트 이전의 정상 쇼트 10개 기준, 24개 변수 z의 rms)
- 경계선: 점수 상위 5% 지점 (**정답을 보지 않고** 정한 선)
- 색과 모양: 회색 동그라미 = 양품 쇼트, 파란 세모 = 쌍둥이(한쪽만 불량), 주황 마름모 = 둘다불량

**비지도가 '잘 됐다'면 보여야 할 것**
1. 대부분의 양품 점은 경계선 **아래**에 모여 있다
2. 경계선 위로 올라온 점 중에 불량이 **우연보다 많다**
3. 올라온 불량이 왜 올라왔는지 **공정 원리로 설명**된다

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from src.official import *
from src.plotting import setup, save
setup()

ML = pd.read_csv(RAW / "moldset_labeled.csv"); MLI = ML.set_index("Unnamed: 0")   # [외부]
GOOD, TWIN, BOTH, INK = "#b4b2a9", "#2a78d6", "#eb6834", "#52514e"
STYLE = {"양품": dict(c=GOOD, marker="o", s=10, alpha=0.6, lw=0),
         "쌍둥이 (한쪽 불량)": dict(c=TWIN, marker="^", s=46, edgecolors="white", linewidths=0.8),
         "둘다불량": dict(c=BOTH, marker="D", s=52, edgecolors="white", linewidths=0.8)}


def shot_scores(p, K=10):
    """쇼트 단위 표: 시각, 불량 종류, 과거기준선 점수. 그리고 변수별 z(부호 포함), 원값."""
    lab = load(p)[0]
    lab["ts"] = pd.to_datetime(lab["order"].map(MLI["TimeStamp"]))
    u = pd.read_csv(RAW / f"unlabeled_raw_{p.lower()}.csv"); u["ts"] = pd.to_datetime(u["TimeStamp"])
    u = u[u["Barrel_Temperature_1"] > 100].drop_duplicates("ts").sort_values("ts")
    g = (lab.groupby("pair").agg(order=("order", "first"), ts=("ts", "first"), bad=("y", "sum"))
            .sort_values("ts").reset_index())
    X = MLI.loc[g["order"], VARS].values
    bt, BX = u["ts"].values, u[VARS].values
    Z = np.zeros_like(X)
    for i, (x, t) in enumerate(zip(X, g["ts"].values)):
        P = BX[bt < t][-K:]
        med = np.median(P, axis=0)
        sd = np.maximum(np.median(np.abs(P - med), axis=0) * 1.4826, P.std(axis=0)); ok = sd > 1e-6
        Z[i] = np.where(ok, np.clip((x - med) / np.where(ok, sd, 1), -20, 20), 0)
    g["score"] = np.sqrt((Z ** 2).mean(axis=1))
    g["kind"] = np.where(g["bad"] == 2, "둘다불량", np.where(g["bad"] == 1, "쌍둥이 (한쪽 불량)", "양품"))
    g["day"] = g["ts"].dt.strftime("%m/%d")
    # 경계선 = 행(부품) 기준 점수 상위 5% (6단계 예산 규칙과 같음)
    thr = np.quantile(lab["pair"].map(g.set_index("pair")["score"]), 0.95)
    raw = pd.DataFrame(X, columns=VARS)
    return g, Z, thr, raw

D = {p: shot_scores(p) for p in ["CN7", "RG3"]}
for p, (g, Z, thr, raw) in D.items():
    print(p, f"쇼트 {len(g)}개, 경계선 점수 {thr:.2f}")

### 그림 1. 시간 순서대로 찍은 점수
가로 = 쇼트 순서(시간), 세로 = 이상 점수(로그 눈금). 세로 회색선은 날짜가 바뀌는 곳입니다.

> **질문 8-1.** 경계선 위에 올라온 **회색 점**(양품)은 검사하면 무엇이 될까요? CN7에서 주황 마름모는 어디쯤 있을까요?
>
> 내 예상:

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7.5))
for ax, (p, (g, Z, thr, raw)) in zip(axes, D.items()):
    x = np.arange(len(g))
    for k, st in STYLE.items():
        m = (g["kind"] == k).values
        ax.scatter(x[m], g["score"][m], label=k, zorder=2 if k == "양품" else 3, **st)
    ax.axhline(thr, color=INK, ls="--", lw=1)
    ax.text(len(g), thr, "상위 5% 경계 ", va="bottom", ha="right", fontsize=9, color=INK)
    cut = np.where(g["day"].values[1:] != g["day"].values[:-1])[0] + 1
    for c in cut:
        ax.axvline(c - 0.5, color="#d6d4cc", lw=0.8, zorder=1)
    for s, e in zip(np.r_[0, cut], np.r_[cut, len(g)]):
        ax.text((s + e) / 2, g["score"].max() * 1.05, g["day"][s], ha="center", fontsize=9, color=INK)
    ax.set_yscale("log"); ax.set_ylabel("이상 점수 (로그)")
    ax.set_title(f"{p} — 쇼트 {len(g)}개, 시간 순서 (위로 갈수록 평소와 다름)", loc="left", fontsize=11)
    ax.grid(axis="y", alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(loc="upper right", fontsize=9, frameon=False)
axes[1].set_xlabel("쇼트 순서 (시간)")
fig.tight_layout(); save(fig, "1002_점수_시간순"); plt.show()

### 그림 2. 종류별로 모아 보기 — 경계선 위로 몇 개가 올라왔나
점마다 좌우로 살짝 흩뿌려 겹치지 않게 했습니다. 숫자 = 경계선 위 개수 / 전체 개수.

> **질문 8-2.** 양품은 약 5%가 경계선 위에 있습니다(경계선 정의상 당연). 쌍둥이가 경계선 위에 있는 비율이 양품과 비슷하다면 무슨 뜻일까요?
>
> 내 예상:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
rng = np.random.RandomState(0)
for ax, (p, (g, Z, thr, raw)) in zip(axes, D.items()):
    for j, (k, st) in enumerate(STYLE.items()):
        v = g.loc[g["kind"] == k, "score"]
        if len(v) == 0:
            continue
        ax.scatter(j + rng.uniform(-0.25, 0.25, len(v)), v, **st)
        ax.text(j, thr * 1.3, f"{(v > thr).sum()}/{len(v)}  ({(v > thr).mean():.0%})", ha="center", fontsize=10)
    ax.axhline(thr, color=INK, ls="--", lw=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(list(STYLE)); ax.set_yscale("log")
    ax.set_title(f"{p} — 경계선 위로 올라온 쇼트", loc="left", fontsize=11)
    ax.grid(axis="y", alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("이상 점수 (로그)")
fig.tight_layout(); save(fig, "1002_점수_종류별"); plt.show()

### 그림 3. 왜 올라왔나 — 충전 시간 vs 사출 속도 (CN7, 원래 단위)
7단계에서 둘다불량 3쇼트는 'R1 충전 시간 × 사출 속도' 관계가 크게 깨졌습니다. 그 관계를 그대로 그립니다.
점선은 양품 쇼트로 그은 직선(참고용)입니다.

> **질문 8-3.** 정상이라면 사출 속도가 빨라질수록 충전 시간은 어떻게 될까요? 주황 마름모는 그 선에서 위로 벗어날까요, 아래로 벗어날까요?
>
> 내 예상:

In [ ]:
g, Z, thr, raw = D["CN7"]
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for k, st in STYLE.items():
    m = (g["kind"] == k).values
    ax.scatter(raw["Max_Injection_Speed"][m], raw["Filling_Time"][m], label=k, **st)
good = (g["kind"] == "양품").values
a, b = np.polyfit(raw["Max_Injection_Speed"][good], raw["Filling_Time"][good], 1)
xs = np.linspace(raw["Max_Injection_Speed"].min(), raw["Max_Injection_Speed"].max(), 50)
ax.plot(xs, a * xs + b, color=INK, ls=":", lw=1, label="양품으로 그은 직선")
ax.set_xlabel("최대 사출 속도 (원값)"); ax.set_ylabel("충전 시간 (초)")
ax.set_title("CN7 — 충전 시간 vs 사출 속도", loc="left", fontsize=11)
ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False); ax.legend(fontsize=9, frameon=False)
fig.tight_layout(); save(fig, "1002_CN7_충전시간_사출속도"); plt.show()

both = (g["kind"] == "둘다불량").values
print("둘다불량 3쇼트: 사출 속도", raw.loc[both, "Max_Injection_Speed"].round(1).tolist(),
      " 충전 시간", raw.loc[both, "Filling_Time"].round(2).tolist(),
      " 직선이 예상한 충전 시간", (a * raw.loc[both, "Max_Injection_Speed"] + b).round(2).tolist())

### 그림 4. 24개 변수를 2차원 지도로 (PCA)
쇼트마다 24개 z를 양품 쇼트 기준 주성분 2개로 눌러 평면에 찍었습니다.
축 이름 옆 %는 그 축이 원래 흩어짐의 몇 %를 담는지입니다.

> **질문 8-4.** CN7은 두 축이 합쳐 약 29%만 담습니다. 이 지도에서 겹쳐 보이는 두 점은 24차원에서도 가깝다고 말할 수 있을까요?
>
> 내 답:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, (p, (g, Z, thr, raw)) in zip(axes, D.items()):
    pc = PCA(2).fit(Z[(g["kind"] == "양품").values])
    Y = pc.transform(Z)
    for k, st in STYLE.items():
        m = (g["kind"] == k).values
        ax.scatter(Y[m, 0], Y[m, 1], label=k, **st)
    r = pc.explained_variance_ratio_
    ax.set_xlabel(f"주성분 1 ({r[0]:.0%})"); ax.set_ylabel(f"주성분 2 ({r[1]:.0%})")
    ax.set_title(f"{p} — 24개 변수 z의 2차원 지도", loc="left", fontsize=11)
    ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(fontsize=9, frameon=False)
fig.tight_layout(); save(fig, "1002_PCA지도"); plt.show()

### 8단계 정리 (직접 적기)
1. 비지도 점수는 양품 대부분을 경계선 아래에 두었나?
2. 경계선 위로 올라온 불량은 어떤 종류였고, 그림 3으로 그 이유를 설명할 수 있나?
3. 쌍둥이와 RG3 불량은 양품과 구별되었나? (그림 2의 비율 비교)
4. RG3 그림 1에서 10/23 후반에 점수가 한꺼번에 올라가는 구간은 무엇일까? (힌트: 불량과 상관없이 양품도 같이 올라감)